# Olist E-Commerce Data Cleaning & Transformation using Python (Pandas)

## Project Objective

This project prepares the raw Olist Brazilian E-Commerce dataset for business analytics by assessing data quality, cleaning inconsistencies, transforming data types, engineering useful features, and integrating related datasets into an analytics-ready format for Power BI.

In [ ]:
# ==================================================
# Import Required Libraries
# ==================================================

import pandas as pd
import numpy as np

## Load the Datasets

The Olist dataset consists of multiple related tables representing customers, orders, products, sellers, payments, reviews, and geolocation data. These datasets will be loaded into separate Pandas DataFrames for further analysis and transformation.

In [ ]:
# ==================================================
# Load Datasets
# ==================================================

orders = pd.read_csv('orders.csv')
order_items = pd.read_csv('order_items.csv')
order_payments = pd.read_csv('order_payments.csv')
order_reviews = pd.read_csv('order_reviews.csv')
customers = pd.read_csv('customers.csv')
products = pd.read_csv('products.csv')
sellers = pd.read_csv('sellers.csv')
geolocation = pd.read_csv('geolocation.csv')
product_category_name_translation = pd.read_csv('product_category_name_translation.csv')

In [ ]:
# ==================================================
# Verifying if Datasets have been loaded successfully
# ==================================================

datasets = {
  'Orders': orders,
  'Order Items': order_items,
  'Order Payments': order_payments,
  'Order Reviews': order_reviews,
  'Customers': customers,
  'Products': products,
  'Sellers': sellers,
  'Geolocation': geolocation,
  'Product Category Translation': product_category_name_translation
}

for name, df in datasets.items():
  print(f"{name}: {df.shape}")

## Dataset Overview

The Olist Brazilian E-Commerce dataset consists of multiple related tables representing customers, orders, products, sellers, payments, reviews, and geolocation information.

Each dataset serves a specific purpose within the overall business process. Before performing any cleaning or transformation, a high-level overview of all datasets is created to understand their size and structure.

In [ ]:
# ==================================================
# Dataset Overview
# ==================================================

summary = pd.DataFrame({
    'Dataset': datasets.keys(),
    'Rows': [df.shape[0] for df in datasets.values()],
    'Columns': [df.shape[1] for df in datasets.values()],
    'Missing Values': [df.isnull().sum().sum() for df in datasets.values()]
})

summary

## Initial Data Profiling

Before cleaning the data, each dataset is profiled to identify potential quality issues.

The following checks are performed:

- Number of records
- Number of columns
- Duplicate rows
- Missing values

This provides an initial understanding of where data quality improvements may be required.

In [ ]:
# ==================================================
# Initial Data Profiling
# ==================================================

profile = pd.DataFrame({
    'Dataset': datasets.keys(),
    'Rows': [df.shape[0] for df in datasets.values()],
    'Columns': [df.shape[1] for df in datasets.values()],
    'Duplicate Rows': [df.duplicated().sum() for df in datasets.values()],
    'Missing Values': [df.isnull().sum().sum() for df in datasets.values()]
})

profile

## Data Quality Assessment

| Dataset | Quality Issue | Business Impact | Cleaning Decision |
|----------|---------------|-----------------|-------------------|
| Orders | Date columns stored as object | Prevents date analysis | Convert to datetime |
| Orders | Missing delivery timestamps | Valid for cancelled/unavailable orders | Retain |
| Products | Missing category information | Incomplete product metadata | Investigate |
| Reviews | Missing review comments | Optional customer feedback | Retain |

## Data Cleaning

The purpose of this section is to improve the quality of the datasets before analysis and reporting.

The following cleaning activities will be performed:

- Convert incorrect data types
- Handle missing values where appropriate
- Remove duplicate records (if any)
- Validate key columns
- Preserve business-valid missing values

In [ ]:
# ==================================================
# Identify Date Columns
# ==================================================

orders.dtypes

In [ ]:
# ==================================================
# Convert Date Columns to Datetime
# ==================================================

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

In [ ]:
# Verify the updated data types

orders[date_columns].dtypes

### Orders Dataset

After converting the timestamp columns to the appropriate data type, the missing values are investigated to determine whether they represent genuine data quality issues or valid business scenarios.

In [ ]:
# ==================================================
# Missing Values in Orders Dataset
# ==================================================

orders.isnull().sum()

In [ ]:
# ==================================================
# Analyze Missing Values by Order Status
# ==================================================

missing_approved = (
    orders[orders["order_approved_at"].isnull()]
    ["order_status"]
    .value_counts()
)

missing_carrier = (
    orders[orders["order_delivered_carrier_date"].isnull()]
    ["order_status"]
    .value_counts()
)

missing_customer = (
    orders[orders["order_delivered_customer_date"].isnull()]
    ["order_status"]
    .value_counts()
)

print("Missing Approval Timestamp")
display(missing_approved)

print("Missing Carrier Delivery Timestamp")
display(missing_carrier)

print("Missing Customer Delivery Timestamp")
display(missing_customer)

### Cleaning Decision

The missing timestamp values were examined against the order status.

The analysis showed that the majority of missing values belong to orders that were cancelled, unavailable, created, invoiced, processing, or shipped. These are valid business scenarios rather than data quality issues.

A very small number of delivered orders have missing timestamps. Since these represent less than 0.02% of the dataset, the records are retained and documented as minor data quality issues.

No missing timestamp values were imputed or removed.

### Validate Key Columns

The `orders` dataset uses `order_id` as its primary key.

Before integrating this dataset with other tables, the uniqueness of the primary key is validated to ensure that each row represents exactly one order.

In [ ]:
# ==================================================
# Validate Primary Key
# ==================================================

print(f"Total Records      : {len(orders)}")
print(f"Unique Order IDs   : {orders['order_id'].nunique()}")

In [ ]:
# Check whether the primary key is unique

orders["order_id"].is_unique

### Validation Result

The `order_id` column contains unique values for every record, confirming that it can safely be used as the primary key when joining the dataset with related tables.

### Cleaning Log

| Step | Dataset | Issue Identified | Action Taken | Status |
|------|---------|------------------|--------------|--------|
| 1 | Orders | Timestamp columns stored as text | Converted to `datetime` | ✅ |
| 2 | Orders | Missing timestamps | Investigated and retained (business-valid) | ✅ |
| 3 | Orders | Primary key validation | Verified `order_id` is unique | ✅ |

## Products Dataset

The `products` dataset contains product-related information such as category, dimensions, weight, and descriptive attributes.

This section assesses the data quality of the dataset and determines whether any cleaning or transformation is required before integrating it with other datasets.

In [ ]:
# ==================================================
# Products Dataset Profile
# ==================================================

print(f"Rows      : {products.shape[0]}")
print(f"Columns   : {products.shape[1]}")
print(f"Duplicates: {products.duplicated().sum()}")

products.isnull().sum()

In [ ]:
# Products with missing category information

products[products["product_category_name"].isnull()].head()

### Business Impact Analysis

The `products` dataset contains missing catalog information for some products.

Before deciding how to handle these missing values, it is important to determine whether these products were actually sold. If they were never purchased, the missing information has little impact on business reporting. If they were sold, further investigation is required.

In [ ]:
# ==================================================
# Products with Missing Category Information
# ==================================================

missing_products = products[
    products["product_category_name"].isnull()
]

print(f"Products with missing category: {len(missing_products)}")

In [ ]:
affected_orders = order_items.merge(
    missing_products[["product_id"]],
    on="product_id",
    how="inner"
)

affected_orders.head()

In [ ]:
print(f"Affected Order Items : {len(affected_orders)}")
print(f"Affected Orders      : {affected_orders['order_id'].nunique()}")
print(f"Affected Products    : {affected_orders['product_id'].nunique()}")

### Cleaning Decision

The products with missing category information were further investigated by comparing them with the `order_items` dataset.

The analysis showed that all 610 products were sold and appear in 1,603 order items across 1,451 customer orders.

Since the correct product categories cannot be inferred reliably, the missing values are retained rather than replaced with artificial values such as "Unknown". This preserves the integrity of the original dataset.

| Step | Dataset  | Issue Identified             | Action Taken                                             | Status |
| ---- | -------- | ---------------------------- | -------------------------------------------------------- | ------ |
| 4    | Products | Missing category information | Investigated business impact and retained missing values | ✅      |


In [ ]:
products[
    products["product_category_name"].isnull()
][
    [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].head(10)

### Observation

The same 610 products are missing the following catalog attributes:

- Product Category
- Product Name Length
- Product Description Length
- Product Photos Quantity

This indicates that the missing values are not isolated issues but represent incomplete product catalog records.

Further analysis showed that these products appear in 1,603 order items across 1,451 customer orders. Since the missing values cannot be inferred reliably, they are retained to preserve the integrity of the original data.

# Data Integration

The Olist dataset consists of multiple related tables. To prepare the data for Power BI reporting, the required datasets are integrated into a single analytics-ready dataset.

Only the columns relevant for business analysis are retained during the integration process.

### Merge Orders and Customers

The `orders` dataset is merged with the `customers` dataset to enrich each order with customer information such as the unique customer identifier, city, and state.

A left join is used to ensure that all orders are retained.

In [ ]:
# ==================================================
# Merge Orders and Customers
# ==================================================

analytics_df = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

analytics_df.shape

### Merge Order Items

Each order may contain one or more products. Therefore, the `order_items` dataset is merged with the existing dataset to include product, seller, price, and freight information.

Since one order can have multiple items, an increase in the number of rows is expected after the merge.

In [ ]:
# ==================================================
# Merge Order Items
# ==================================================

analytics_df = analytics_df.merge(
    order_items,
    on="order_id",
    how="left"
)

In [ ]:
print(f"Shape            : {analytics_df.shape}")
print(f"Unique Orders    : {analytics_df['order_id'].nunique()}")
print(f"Unique Customers : {analytics_df['customer_id'].nunique()}")

**Validation Result**

- All 99,441 orders were successfully retained after the merge.
- The number of rows increased as expected because an order can contain multiple products.
- The merged dataset is now at the **order-item level**, where each row represents a single item within an order.

### Merge Products

The `products` dataset is merged with the analytics dataset to enrich each order item with product attributes such as category, weight, and dimensions.

A left join is used to preserve all order items, even if some product metadata is missing.

In [ ]:
# ==================================================
# Merge Products
# ==================================================

analytics_df = analytics_df.merge(
    products,
    on="product_id",
    how="left"
)

In [ ]:
print(f"Shape            : {analytics_df.shape}")
print(f"Unique Orders    : {analytics_df['order_id'].nunique()}")
print(f"Unique Products  : {analytics_df['product_id'].nunique()}")

**Validation Result**

- The merge preserved all order items.
- Product attributes were successfully added.
- Missing product metadata remains unchanged and has been retained as documented during the data quality assessment.

### Translate Product Categories

The product category names are translated from Portuguese to English using the category translation dataset.

This improves readability for reporting and makes the analytics dataset easier to understand.

In [ ]:
# ==================================================
# Translate Product Categories
# ==================================================

analytics_df = analytics_df.merge(
    product_category_name_translation,
    on="product_category_name",
    how="left"
)

In [ ]:
analytics_df[
    [
        "product_category_name",
        "product_category_name_english"
    ]
].head()

### Merge Sellers

The `sellers` dataset is merged to enrich each order item with seller location information.

This enables geographical analysis of seller performance in Power BI.

In [ ]:
# ==================================================
# Merge Sellers
# ==================================================

analytics_df = analytics_df.merge(
    sellers,
    on="seller_id",
    how="left"
)

In [ ]:
print(f"Shape           : {analytics_df.shape}")
print(f"Unique Sellers  : {analytics_df['seller_id'].nunique()}")

### Aggregate Payment Information

An order may have multiple payment transactions. To avoid duplicating rows during the merge, the payment information is aggregated to the order level before integration.

In [ ]:
# ==================================================
# Aggregate Payments
# ==================================================

payments_summary = (
    order_payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max")
    )
)

payments_summary.head()

In [ ]:
analytics_df = analytics_df.merge(
    payments_summary,
    on="order_id",
    how="left"
)

In [ ]:
print(f"Shape : {analytics_df.shape}")

### Merge Reviews

Customer review information is merged to include the review score for each order.

For this analytics dataset, only the review score is retained because textual review comments are not required for dashboard reporting.

In [ ]:
review_summary = order_reviews[
    ["order_id", "review_score"]
]

analytics_df = analytics_df.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [ ]:
print(f"Final Shape : {analytics_df.shape}")
print(f"Unique Orders : {analytics_df['order_id'].nunique()}")

In [ ]:
# Check Point (Reusability)

analytics_df.to_csv(
    "olist_integrated_dataset.csv",
    index=False
)

## Feature Engineering

The integrated dataset is enhanced by creating new business-friendly columns that simplify reporting and improve dashboard performance.

These engineered features eliminate the need for repeated calculations inside Power BI and make the dataset analytics-ready.

In [ ]:
# ==================================================
# Purchase Year
# ==================================================

analytics_df["purchase_year"] = (
    analytics_df["order_purchase_timestamp"].dt.year
)

analytics_df["purchase_month"] = (
    analytics_df["order_purchase_timestamp"].dt.month
)

analytics_df["purchase_month_name"] = (
    analytics_df["order_purchase_timestamp"]
    .dt.month_name()
)

analytics_df["purchase_quarter"] = (
    analytics_df["order_purchase_timestamp"]
    .dt.quarter
)

analytics_df["purchase_day"] = (
    analytics_df["order_purchase_timestamp"]
    .dt.day_name()
)

analytics_df["delivery_days"] = (
    analytics_df["order_delivered_customer_date"]
    -
    analytics_df["order_purchase_timestamp"]
).dt.days

analytics_df["estimated_delivery_days"] = (
    analytics_df["order_estimated_delivery_date"]
    -
    analytics_df["order_purchase_timestamp"]
).dt.days

analytics_df["delivery_delay_days"] = (
    analytics_df["order_delivered_customer_date"] - analytics_df["order_estimated_delivery_date"]
).dt.days


analytics_df["late_delivery"] = (
    analytics_df["delivery_delay_days"] > 0
)

analytics_df["total_item_value"] = (
    analytics_df["price"]
    +
    analytics_df["freight_value"]
)

analytics_df["freight_percentage"] = (
    analytics_df["freight_value"] / analytics_df["price"]
) * 100

order_size = (
    analytics_df
    .groupby("order_id")
    .size()
    .reset_index(name="order_size")
)

In [ ]:
analytics_df.info()

## Export Analytics Dataset

The cleaned, integrated, and feature-engineered dataset is exported for visualization and dashboard development in Power BI.

In [ ]:
analytics_df.to_csv(
    "olist_analytics_ready.csv",
    index=False
)

# Project Summary

This project transformed the Olist Brazilian E-Commerce dataset into an analytics-ready dataset for business reporting.

### Key Activities
- Imported and profiled multiple datasets.
- Performed data quality assessment.
- Investigated and documented missing values.
- Converted data types.
- Integrated related datasets using appropriate joins.
- Engineered business-focused features.
- Exported a single analytics-ready dataset for Power BI.

### Skills Demonstrated
- Python
- Pandas
- Data Cleaning
- Data Transformation
- Feature Engineering
- Data Integration
- Business Analysis
- Data Validation